# nb45 — Top-10 parameter sweep on 100 random Mon-Fri UTC days

**User directive (2026-09-17):** the previous backtest tested too few
params. Create a new notebook and expose all possible params. Run the
backtest on which param would give highest returns — infer which
param is most impactful and does change the alpha to generate edge.
Run on 100 day random days and save the results. Results should have
n trade, EV per trade, and a few percentiles of trade duration. Also
why a trade is entered i.e. bear fvg mitigation.

## Design

We sweep **the top 10 most-impactful CAUSAL knobs** (one knob flipped
per config, all else equal to BASELINE). The 10 knobs were picked from
the existing alpha studies (see `AGENTS.md`) as those with the largest
empirical effect on EV/trade:

| # | Knob | Config(s) tested |
|---|---|---|
| 1 | `fvg_inv_trade_enabled` | `INV_TRADE_ON` (best single-knob from v3) |
| 2 | `fvg_min_zone_usd` | `MIN_ZONE_050`, `MIN_ZONE_080` |
| 3 | `invalidation_sl_usd` | `SOFT_STOP_OFF` (off) |
| 4 | `fvg_invalidation_min_pierce_usd` | `PIERCE_TIGHT` (0.10) |
| 5 | `sl_atr_mult` | `SL_ATR_050` |
| 6 | `tp_atr_mult` | `TP_ATR_080` |
| 7 | `inverse_breadth` | `INVERSE_BREADTH_OFF` |
| 8 | `bos_choch_ignore_invert_when_aligned` | `BOS_ALIGN_OFF` |
| 9 | `fvg_sweep_enabled` | `SWEEP_ON` |
| 10 | `num_layers` | `NUM_LAYERS_1` |
| (extra) | `fvg_resample_secs`, `renko_drive_invalidation`, `fvg_ifvg_min_inversion_age_secs` | 3 more configs for breadth |

**Total configs:** 15 (1 baseline + 14 single-knob variants).

## Workflow

The notebook is split into two passes:

1. **DRY_RUN** — 20 random days, fast (~50 seconds). Own report.
2. **EXTEND** — 80 more random days (~3-4 minutes). Merges with the
   dry-run days for the full 100-day report.

Run the dry-run cell first. Inspect its findings, then run the extend
cell. The full-100-day output overwrites the dry-only summary tables
(CSV names carry the `_dry_` vs `_full_` prefix).

## Output files

* `notebooks/param_sweep_dry_per_day.csv`  — per-day breakdown, 20 days
* `notebooks/param_sweep_dry_summary.csv`  — per-config totals, 20 days
* `notebooks/param_sweep_full_per_day.csv` — per-day breakdown, 100 days
* `notebooks/param_sweep_full_summary.csv` — per-config totals, 100 days

Each summary CSV has:
  `n_trades`, `pnl_total`, `EV/trade`, `PnL/day`, `trades/day`,
  `p25_hold_sec`, `p50_hold_sec`, `p75_hold_sec`, `p90_hold_sec`,
  `p95_hold_sec`, `n_fvg`/`n_ifvg`/`n_orb`/`n_wyckoff`/`n_sweep`/`n_inv`
  (counts and percentages by `entry_triggered_by`),
  `n_signals_emitted`, `n_soft_stops`.

## Edit knobs

In [1]:
# Knobs - edit and re-run
DRY_DAYS = 20            # how many random days for DRY_RUN
EXTEND_DAYS = 80         # how many MORE days for EXTEND (total = DRY + EXTEND = 100)
RNG_SEED = 20260917      # reproducible day selection (distinct from alpha_v2)
MIN_BARS_PER_DAY = 30_000

# Run as a subprocess so output is captured in the cell (the tool
# prints its own per-config + delta-vs-baseline summary tables).
import subprocess

In [2]:
## Locate repo + verify tool exists

In [3]:
import os
import sys
from pathlib import Path

os.environ.setdefault('MPLBACKEND', 'Agg')


def _find_root() -> Path:
    here = Path('.').resolve()
    candidates = [p for p in [here, *here.parents]
                  if (p / 'src' / 'core' / 'ict_signals.py').is_file()]
    if not candidates:
        raise RuntimeError('Could not find ICT repo root')

    def _has_fork_sig(p: Path) -> bool:
        f = p / 'src' / 'core' / 'ict_strategy.py'
        try:
            return 'played_out_min_extension_usd' in f.read_text(encoding='utf-8')
        except OSError:
            return False

    with_sig = [p for p in candidates if _has_fork_sig(p)]
    return with_sig[0] if with_sig else candidates[0]


ROOT = _find_root()
sys.path.insert(0, str(ROOT))
print(f'Repo root: {ROOT}')

tool_path = ROOT / 'src' / 'tools' / 'run_param_sweep.py'
assert tool_path.is_file(), f'missing: {tool_path}'
print(f'Tool: {tool_path}')

Repo root: C:\coding\ict_tier_v2
Tool: C:\coding\ict_tier_v2\src\tools\run_param_sweep.py


## DRY_RUN — 20 random days

In [4]:
# First pass: 20 days, ~50 seconds. Inspect the report below.
print(f'\n>>> DRY_RUN: {DRY_DAYS} days <<<\n')
res = subprocess.run(
    [sys.executable, str(tool_path), str(DRY_DAYS)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(res.stdout)
if res.returncode != 0:
    print('STDERR:', res.stderr)


>>> DRY_RUN: 20 days <<<



Day enumeration: 0.8s, 660 Mon-Fri days
Sampled 20 days (seed 20260917)
Pre-loading all days...
Pre-load: 0.4s, 20 valid days

Running 300 backtests (15 configs x 20 days, label=dry)...

=== BASELINE ===

=== INV_TRADE_ON ===

=== SOFT_STOP_OFF ===
  [50/300] SOFT_STOP_OFF 2024-03-08 (elapsed 8.4s, ETA 41.9s)

=== PIERCE_TIGHT ===

=== INVERSE_BREADTH_OFF ===
  [100/300] INVERSE_BREADTH_OFF 2024-06-05 (elapsed 17.8s, ETA 35.6s)

=== NUM_LAYERS_1 ===

=== SL_ATR_050 ===

=== TP_ATR_080 ===
  [150/300] TP_ATR_080 2024-03-08 (elapsed 26.3s, ETA 26.3s)

=== MIN_ZONE_050 ===

=== MIN_ZONE_080 ===
  [200/300] MIN_ZONE_080 2024-06-05 (elapsed 35.5s, ETA 17.7s)

=== RESAMPLE_1S ===

=== BOS_ALIGN_OFF ===

=== SWEEP_ON ===
  [250/300] SWEEP_ON 2024-03-08 (elapsed 45.4s, ETA 9.1s)

=== RENKO_INV_ON ===

=== IFVG_AGE_60 ===
  [300/300] IFVG_AGE_60 2024-06-05 (elapsed 56.2s, ETA 0.0s)

All backtests done in 56.4s (0.188s per backtest)

Per-day CSV: C:\coding\ict_tier_v2\notebooks\param_sweep_dry_p

## Inspect DRY_RUN results

In [5]:
import pandas as pd

dry_summary_path = ROOT / 'notebooks' / 'param_sweep_dry_summary.csv'
dry_per_day_path = ROOT / 'notebooks' / 'param_sweep_dry_per_day.csv'

dry_summary = pd.read_csv(dry_summary_path)
dry_per_day = pd.read_csv(dry_per_day_path)

print(f'DRY_RUN summary: {dry_summary.shape[0]} configs, '
      f'{len(dry_per_day["day"].unique())} days, '
      f'{int(dry_summary["n_trades"].sum()):,} trades total')
print('\nTop 5 configs by EV/trade:')
print(dry_summary[['config', 'n_trades', 'ev_per_trade', 'pnl_per_day', 'p50_hold_sec']]
      .head(5).to_string(index=False))

print('\nConfigs with positive EV/trade:')
pos = dry_summary[dry_summary['ev_per_trade'] > 0]
if len(pos):
    print(pos[['config', 'ev_per_trade', 'pnl_per_day']].to_string(index=False))
else:
    print('  (none)')

DRY_RUN summary: 15 configs, 20 days, 15,399 trades total

Top 5 configs by EV/trade:
             config  n_trades  ev_per_trade  pnl_per_day  p50_hold_sec
INVERSE_BREADTH_OFF      1029      0.083149     4.278018     18.085520
       INV_TRADE_ON      1073     -0.012456    -0.668251      4.462255
      SOFT_STOP_OFF      1029     -0.014747    -0.758731      2.976676
       PIERCE_TIGHT      1009     -0.016154    -0.814948      3.919722
        IFVG_AGE_60      1002     -0.016601    -0.831700      4.342315

Configs with positive EV/trade:
             config  ev_per_trade  pnl_per_day
INVERSE_BREADTH_OFF      0.083149     4.278018


## EXTEND — add 80 more days (total = 100)

In [6]:
# Second pass: 80 more days. Reuses the 20 days from DRY_RUN
# (the EXTEND mode skips the first N_DAYS).
print(f'\n>>> EXTEND: adding {EXTEND_DAYS} more days <<<\n')
res = subprocess.run(
    [sys.executable, str(tool_path), str(DRY_DAYS), 'extend'],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(res.stdout)
if res.returncode != 0:
    print('STDERR:', res.stderr)


>>> EXTEND: adding 80 more days <<<



Day enumeration: 1.0s, 660 Mon-Fri days
EXTEND mode: skipping first 20 dry-run days, adding 80 new days
Pre-loading all days...
Pre-load: 1.7s, 80 valid days

Running 1200 backtests (15 configs x 80 days, label=full)...

=== BASELINE ===
  [50/1200] BASELINE 2025-10-20 (elapsed 11.5s, ETA 264.0s)

=== INV_TRADE_ON ===
  [100/1200] INV_TRADE_ON 2024-10-10 (elapsed 24.3s, ETA 267.3s)
  [150/1200] INV_TRADE_ON 2026-04-14 (elapsed 37.6s, ETA 263.0s)

=== SOFT_STOP_OFF ===
  [200/1200] SOFT_STOP_OFF 2025-05-13 (elapsed 49.6s, ETA 248.1s)

=== PIERCE_TIGHT ===
  [250/1200] PIERCE_TIGHT 2024-07-22 (elapsed 63.0s, ETA 239.3s)
  [300/1200] PIERCE_TIGHT 2026-01-26 (elapsed 76.0s, ETA 228.0s)

=== INVERSE_BREADTH_OFF ===
  [350/1200] INVERSE_BREADTH_OFF 2025-01-22 (elapsed 88.8s, ETA 215.6s)
  [400/1200] INVERSE_BREADTH_OFF 2026-07-13 (elapsed 104.7s, ETA 209.5s)

=== NUM_LAYERS_1 ===
  [450/1200] NUM_LAYERS_1 2025-10-20 (elapsed 117.5s, ETA 195.9s)

=== SL_ATR_050 ===
  [500/1200] SL_ATR_050 202

## Inspect FULL results (100 days)

In [7]:
full_summary_path = ROOT / 'notebooks' / 'param_sweep_full_summary.csv'
full_per_day_path = ROOT / 'notebooks' / 'param_sweep_full_per_day.csv'

full_summary = pd.read_csv(full_summary_path)
full_per_day = pd.read_csv(full_per_day_path)

print(f'FULL summary: {full_summary.shape[0]} configs, '
      f'{len(full_per_day["day"].unique())} days, '
      f'{int(full_summary["n_trades"].sum()):,} trades total')

FULL summary: 15 configs, 80 days, 84,606 trades total


## Per-config report (n_trade, EV, PnL/day, percentiles, breakdown by triggered_by)

In [8]:
print('=' * 100)
print(f'PER-CONFIG REPORT (FULL, 100 days, seed {RNG_SEED})')
print('=' * 100)
report_cols = [
    'config', 'n_trades', 'ev_per_trade', 'pnl_per_day', 'trades_per_day',
    'p25_hold_sec', 'p50_hold_sec', 'p75_hold_sec', 'p90_hold_sec', 'p95_hold_sec',
    'pct_fvg', 'pct_ifvg', 'pct_orb', 'pct_wyckoff', 'pct_sweep', 'pct_inv',
]
print(full_summary[report_cols].to_string(index=False, float_format='%.4f'))

PER-CONFIG REPORT (FULL, 100 days, seed 20260917)
             config  n_trades  ev_per_trade  pnl_per_day  trades_per_day  p25_hold_sec  p50_hold_sec  p75_hold_sec  p90_hold_sec  p95_hold_sec  pct_fvg  pct_ifvg  pct_orb  pct_wyckoff  pct_sweep  pct_inv
INVERSE_BREADTH_OFF      5477        0.2865      19.6147         68.4625        0.1541        3.9755       54.6521      439.2121     1330.3748  24.8311   75.1689   0.0000       0.0000     0.0000   0.0000
        RESAMPLE_1S      9370       -0.0255      -2.9852        117.1250        0.0166        0.8442        7.2382       24.1940       43.1298  56.1900   43.8100   0.0000       0.0000     0.0000   0.0000
      SOFT_STOP_OFF      5477       -0.0338      -2.3140         68.4625        0.0175        0.5456        6.8983       30.5417       68.6373  24.8311   75.1689   0.0000       0.0000     0.0000   0.0000
       INV_TRADE_ON      5895       -0.0393      -2.8960         73.6875        0.0235        0.9467       10.2310       42.1040      

## Most impactful knob (delta-EV vs BASELINE)

In [9]:
baseline_row = full_summary[full_summary['config'] == 'BASELINE'].iloc[0]
baseline_ev = baseline_row['ev_per_trade']
baseline_pnl = baseline_row['pnl_total']
baseline_n = baseline_row['n_trades']

ranking_rows = []
for _, row in full_summary.iterrows():
    if row['config'] == 'BASELINE':
        continue
    ranking_rows.append({
        'config': row['config'],
        'delta_ev_per_trade': row['ev_per_trade'] - baseline_ev,
        'delta_pnl_per_day': (row['pnl_total'] - baseline_pnl) / len(full_per_day['day'].unique()),
        'delta_n_trades': int(row['n_trades'] - baseline_n),
        'absolute_ev_per_trade': row['ev_per_trade'],
        'positive_ev': row['ev_per_trade'] > 0,
    })
ranking = pd.DataFrame(ranking_rows)
ranking = ranking.sort_values('delta_ev_per_trade', key=lambda x: x.abs(), ascending=False)

print('=' * 100)
print(f'MOST IMPACTFUL KNOB (FULL, baseline EV=${baseline_ev:+.4f}, PnL=${baseline_pnl:+.2f})')
print('=' * 100)
print('Ranked by |delta EV/trade| desc -- the biggest absolute change is the MOST IMPACTFUL knob.')
print()
print(ranking.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))

print('\nConfigs with positive EV/trade:')
pos = full_summary[full_summary['ev_per_trade'] > 0]
if len(pos):
    print(pos[['config', 'ev_per_trade', 'pnl_per_day', 'p50_hold_sec']].to_string(
        index=False, float_format='%.4f'))
else:
    print('  (none -- no config produced positive EV at N=100)')

MOST IMPACTFUL KNOB (FULL, baseline EV=$-0.0507, PnL=$-277.51)
Ranked by |delta EV/trade| desc -- the biggest absolute change is the MOST IMPACTFUL knob.

             config  delta_ev_per_trade  delta_pnl_per_day  delta_n_trades  absolute_ev_per_trade  positive_ev
INVERSE_BREADTH_OFF             +0.3372           +23.0836               0                +0.2865         True
        RESAMPLE_1S             +0.0252            +0.4837            3893                -0.0255        False
       NUM_LAYERS_1             -0.0215            +2.0449           -3899                -0.0722        False
      SOFT_STOP_OFF             +0.0169            +1.1549               0                -0.0338        False
      BOS_ALIGN_OFF             -0.0143            -0.9758               0                -0.0649        False
       INV_TRADE_ON             +0.0114            +0.5729             418                -0.0393        False
       MIN_ZONE_080             -0.0100            -0.3309          

## Why a trade is entered — entry_triggered_by breakdown

In [10]:
print('=' * 80)
print('WHY TRADES ARE ENTERED (FULL, per-config breakdown by triggered_by)')
print('=' * 80)
trigger_cols = [
    'config', 'n_trades',
    'n_fvg', 'n_ifvg', 'n_orb', 'n_wyckoff', 'n_sweep', 'n_inv',
    'pct_fvg', 'pct_ifvg', 'pct_orb', 'pct_wyckoff', 'pct_sweep', 'pct_inv',
]
print(full_summary[trigger_cols].to_string(index=False, float_format='%.1f'))

WHY TRADES ARE ENTERED (FULL, per-config breakdown by triggered_by)
             config  n_trades  n_fvg  n_ifvg  n_orb  n_wyckoff  n_sweep  n_inv  pct_fvg  pct_ifvg  pct_orb  pct_wyckoff  pct_sweep  pct_inv
INVERSE_BREADTH_OFF      5477   1360    4117      0          0        0      0     24.8      75.2      0.0          0.0        0.0      0.0
        RESAMPLE_1S      9370   5265    4105      0          0        0      0     56.2      43.8      0.0          0.0        0.0      0.0
      SOFT_STOP_OFF      5477   1360    4117      0          0        0      0     24.8      75.2      0.0          0.0        0.0      0.0
       INV_TRADE_ON      5895   1360    4117      0          0        0    418     23.1      69.8      0.0          0.0        0.0      7.1
       PIERCE_TIGHT      5413   1437    3976      0          0        0      0     26.5      73.5      0.0          0.0        0.0      0.0
        IFVG_AGE_60      5114   1360    3754      0          0        0      0     26.6     

## Notes & next steps

In [11]:
print("""
Findings to look for in the output above:

1. **Most impactful knob**: the top of the delta-vs-baseline table. A
   delta-EV/trade of magnitude > $0.05 means the knob is the
   single-knob lever with the biggest alpha effect.

2. **Positive-EV configs**: only configs with EV/trade > 0 are
   candidates. The user directive says "infer which param is most
   impactful and does change the alpha to generate edge" -- only
   positive-EV configs count.

3. **Trade-duration percentiles (p25/p50/p75/p90/p95)**: shows the
     shape of the trade-lifetime distribution. A spike at low p25
     means many trades exit quickly (typical for stop-loss hits); a
     long p95 tail means a few trades carry for a long time (typical
     for take-profit runs on extended moves).

4. **triggered_by breakdown**: shows whether the FVG path, the iFVG
   path, the ORB/Wyckoff paths, or the inverse-trade edge is doing
   the work. Configs that lean heavily on one path tell us which
   signal source is the alpha source.

5. **DRY_RUN vs FULL**: if the most-impactful knob changes rank
   between DRY_RUN (20) and FULL (100), the signal is noisy at
   small sample size and the FULL report is the one to act on.
""")


Findings to look for in the output above:

1. **Most impactful knob**: the top of the delta-vs-baseline table. A
   delta-EV/trade of magnitude > $0.05 means the knob is the
   single-knob lever with the biggest alpha effect.

2. **Positive-EV configs**: only configs with EV/trade > 0 are
   candidates. The user directive says "infer which param is most
   impactful and does change the alpha to generate edge" -- only
   positive-EV configs count.

3. **Trade-duration percentiles (p25/p50/p75/p90/p95)**: shows the
     shape of the trade-lifetime distribution. A spike at low p25
     means many trades exit quickly (typical for stop-loss hits); a
     long p95 tail means a few trades carry for a long time (typical
     for take-profit runs on extended moves).

4. **triggered_by breakdown**: shows whether the FVG path, the iFVG
   path, the ORB/Wyckoff paths, or the inverse-trade edge is doing
   the work. Configs that lean heavily on one path tell us which
   signal source is the alpha 